In [3]:
# =========================================
# 00 — SETUP: CREATE NECESSARY DIRECTORIES
# =========================================

import os

# Create processed data and plots directories if they don't exist
dirs = ['../data/processed/', '../plots/preprocessing/', '../plots/clustering/', '../plots/classification/', '../models/']
for dir_path in dirs:
    os.makedirs(dir_path, exist_ok=True)
    
print("✓ All directories created/verified")


✓ All directories created/verified


In [4]:
# =========================================
# 02 — PREPROCESSING
# =========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle
import warnings
warnings.filterwarnings('ignore')

# ================================
# LOAD DATA
# ================================
df = pd.read_csv('../data/raw/raw_data.csv')
print("Raw shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())

# ================================
# FEATURE SELECTION & RENAMING
# ================================
print("\n" + "="*60)
print("FEATURE SELECTION & RENAMING")
print("="*60)

# Select key financial and behavioral features
selected_features = {
    'Age': 'Age',
    'AnnualIncome': 'Income',
    'LoanAmount': 'LoanAmount',
    'CreditScore': 'CreditScore',
    'Experience': 'YearsExperience',
    'EmploymentStatus': 'EmploymentType',
    'EducationLevel': 'Education',
    'MaritalStatus': 'MaritalStatus',
    'HomeOwnershipStatus': 'HomeOwnership',
    'DebtToIncomeRatio': 'DebtToIncome',
    'BankruptcyHistory': 'BankruptcyHistory',
    'PaymentHistory': 'PaymentHistory',
}

# Create new dataframe with selected features
X = df[list(selected_features.keys())].copy()
X.columns = list(selected_features.values())

# Add target
y = df['LoanApproved'].copy()

print(f"Selected features: {list(X.columns)}")
print(f"Target: LoanApproved")
print(f"\nData shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")
print(f"Class balance:\n{(y.value_counts() / len(y) * 100).round(2)}")

# ================================
# IDENTIFY COLUMN TYPES
# ================================
numeric_cols = X.select_dtypes(include='number').columns.tolist()
categorical_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()

print(f"\nNumeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")

# ================================
# HANDLE MISSING VALUES
# ================================
print("\n" + "="*60)
print("HANDLING MISSING VALUES")
print("="*60)

# Fix missing values in numeric columns (impute with median)
for col in numeric_cols:
    if X[col].isnull().sum() > 0:
        median_val = X[col].median()
        X[col].fillna(median_val, inplace=True)
        print(f"  ✓ {col}: imputed {X[col].isnull().sum()} values with median {median_val:.2f}")

# Fix missing values in categorical columns (impute with mode)
for col in categorical_cols:
    if X[col].isnull().sum() > 0:
        mode_val = X[col].mode()[0]
        X[col].fillna(mode_val, inplace=True)
        print(f"  ✓ {col}: imputed with mode '{mode_val}'")

print(f"Total missing values: {X.isnull().sum().sum()}")

# ================================
# OUTLIER HANDLING (IQR METHOD)
# ================================
print("\n" + "="*60)
print("HANDLING OUTLIERS (IQR Method)")
print("="*60)

outlier_indices = set()

for col in numeric_cols:
    Q1 = X[col].quantile(0.25)
    Q3 = X[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers_mask = (X[col] < lower_bound) | (X[col] > upper_bound)
    num_outliers = outliers_mask.sum()
    
    if num_outliers > 0:
        outlier_indices.update(np.where(outliers_mask)[0].tolist())
        print(f"  {col}: {num_outliers} outliers (bounds: {lower_bound:.2f} - {upper_bound:.2f})")

outlier_indices = list(outlier_indices)
print(f"\nTotal outlier rows: {len(outlier_indices)}")

# Remove outliers
X_clean = X.drop(X.index[outlier_indices]).reset_index(drop=True)
y_clean = y.drop(y.index[outlier_indices]).reset_index(drop=True)

print(f"Shape after outlier removal: {X_clean.shape}")

# ================================
# ENCODE CATEGORICAL VARIABLES
# ================================
print("\n" + "="*60)
print("ENCODING CATEGORICAL VARIABLES")
print("="*60)

label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    X_clean[col] = le.fit_transform(X_clean[col].astype(str))
    label_encoders[col] = le
    print(f"  ✓ {col}: encoded {len(le.classes_)} unique values")
    print(f"      Classes: {list(le.classes_)}")

# ================================
# SCALE NUMERICAL FEATURES
# ================================
print("\n" + "="*60)
print("SCALING NUMERICAL FEATURES (StandardScaler)")
print("="*60)

scaler = StandardScaler()
X_scaled = X_clean.copy()
X_scaled[numeric_cols] = scaler.fit_transform(X_clean[numeric_cols])

print(f"Scaling applied to {len(numeric_cols)} numeric columns")
print(f"\nScaled data statistics:")
print(X_scaled.describe())

# ================================
# COMBINE AND SAVE PROCESSED DATA
# ================================
print("\n" + "="*60)
print("SAVING PROCESSED DATA")
print("="*60)

# Combine features and target
processed_df = X_scaled.copy()
processed_df['LoanApproved'] = y_clean.values

print(f"Processed data shape: {processed_df.shape}")
print(f"Processed data columns: {list(processed_df.columns)}")

# Save processed data
output_path = '../data/processed/processed_data.csv'
processed_df.to_csv(output_path, index=False)

print(f"✓ Processed data saved to: {output_path}")
print(f"\nFirst 5 rows of processed data:")
print(processed_df.head())

# Save label encoders
encoder_path = '../models/label_encoders.pkl'
with open(encoder_path, 'wb') as f:
    pickle.dump(label_encoders, f)
print(f"✓ Label encoders saved to: {encoder_path}")

# Save scaler
scaler_path = '../models/scaler.pkl'
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f"✓ Scaler saved to: {scaler_path}")

# ================================
# SUMMARY STATISTICS
# ================================
print("\n" + "="*60)
print("PREPROCESSING SUMMARY")
print("="*60)
print(f"Raw data shape: {df.shape}")
print(f"After outlier removal: {processed_df.shape}")
print(f"Rows removed: {df.shape[0] - processed_df.shape[0]}")
print(f"Columns selected: {len(selected_features)}")
print(f"Numeric features: {len(numeric_cols)}")
print(f"Categorical features: {len(categorical_cols)}")
print(f"\nTarget distribution:")
print(processed_df['LoanApproved'].value_counts())
print(f"\nClass balance:")
print((processed_df['LoanApproved'].value_counts() / len(processed_df) * 100).round(2))
print(f"\n✓ PREPROCESSING COMPLETE!")


Raw shape: (20000, 36)

First few rows:
  ApplicationDate  Age  AnnualIncome  CreditScore EmploymentStatus  \
0      2018-01-01   45         39948          617         Employed   
1      2018-01-02   38         39709          628         Employed   
2      2018-01-03   47         40724          570         Employed   
3      2018-01-04   58         69084          545         Employed   
4      2018-01-05   37        103264          594         Employed   

  EducationLevel  Experience  LoanAmount  LoanDuration MaritalStatus  ...  \
0         Master          22       13152            48       Married  ...   
1      Associate          15       26045            48        Single  ...   
2       Bachelor          26       17627            36       Married  ...   
3    High School          34       37898            96        Single  ...   
4      Associate          17        9184            36       Married  ...   

   MonthlyIncome UtilityBillsPaymentHistory  JobTenure  NetWorth  \
0    332